# 3.1.2 — Mapa SHAP: síntese visual e fechamento da Fase 1

Este notebook é o **ponto de fechamento da Fase 1 (SHAP)** da Etapa 3. Consolida em um único artefato navegável as verificações visuais de [PA_3.1 §2](../../NOTAS/PA_3.1.md) e formaliza as decisões A, B, C, D de [PA_3.1 §3](../../NOTAS/PA_3.1.md). A estrutura segue [PLANO_3.1.2](../../NOTAS/PLANO_3.1.2.md).

**Ordem das seções:** panorama (heatmap, cobertura, timing) → detalhe (bar/beeswarm) → narrativa (P1, T2) → painel de síntese → decisões → checklist.

**Não recalcula SHAP.** Reaproveita:
- `ARTEFATOS/ETAPA_3/shap/3.1_shap_consolidado.csv`
- `ARTEFATOS/ETAPA_3/shap/<modelo>_<output>_shap.npy` (15 matrizes 200×8)
- `ARTEFATOS/ETAPA_3/shap/3.1_diag_*.{png,csv}`
- `ARTEFATOS/ETAPA_3/shap/3.1_bar_*.png`, `3.1_beeswarm_*.png`
- MLflow `phase=shap` (15 runs)

**Saídas novas:** `3.1.2_painel_S6.png` e `3.1.2_diagnostico_decisoes.csv`.

## Seção 1 — Setup e carga

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Rectangle

PROJECT_ROOT = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO")
ETAPA_3      = PROJECT_ROOT / "ARTEFATOS" / "ETAPA_3"
SHAP_DIR     = ETAPA_3 / "3.1_SHAP"
SHAP_3_1_DIR = SHAP_DIR                   # consolidado, diag e outputs de 3.1.2
SHAP_NPY_DIR = SHAP_DIR / "artefatos"     # matrizes .npy
SHAP_BAR_DIR = SHAP_DIR / "3.1_BAR"       # bar plots
SHAP_BEE_DIR = SHAP_DIR / "3.1_BEESWARM"  # beeswarm plots
OUT_DIR      = SHAP_DIR                   # saídas de 3.1.2 (painel, csv, md)
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_CONSOL   = SHAP_3_1_DIR / "3.1_shap_consolidado.csv"
MLRUNS_URI   = f"file://{PROJECT_ROOT}/mlruns"

FEATURE_NAMES = ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
MODELS        = ["SVR", "DT", "RF", "XGBoost", "ANN"]
OUTPUTS       = ["ET", "M_CH3OH", "x_CH3OH"]
RANDOM_STATE  = 42

# Hipótese principal vinda de PA_3.1 §1.3 (convergência de 3 critérios)
S_6 = ["T1", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
F_DESCARTADAS = ["P1", "T2"]

assert CSV_CONSOL.exists(), f"Consolidado não encontrado em {CSV_CONSOL}"
assert SHAP_NPY_DIR.exists(), f"Pasta de matrizes .npy não encontrada em {SHAP_NPY_DIR}"
assert set(S_6) | set(F_DESCARTADAS) == set(FEATURE_NAMES)

In [ ]:
df = pd.read_csv(CSV_CONSOL)

# Validações estruturais (idênticas às de 3.1.1)
assert len(df) == len(MODELS) * len(OUTPUTS) * len(FEATURE_NAMES), f"Esperado 120 linhas, obtido {len(df)}"
assert set(df["modelo"].unique())  == set(MODELS)
assert set(df["output"].unique())  == set(OUTPUTS)
assert set(df["feature"].unique()) == set(FEATURE_NAMES)
assert not df.isna().any().any(), "NaN presente no consolidado"

# Carga das 15 matrizes .npy (para usos pontuais; não recomputa SHAP)
shap_matrices = {
    (m, o): np.load(SHAP_NPY_DIR / f"{m}_{o}_shap.npy")
    for m in MODELS for o in OUTPUTS
}
shapes = {k: v.shape for k, v in shap_matrices.items()}
assert all(s[1] == 8 for s in shapes.values()), "Matrizes SHAP com nº de features inesperado"

# Container de diagnóstico que será populado ao longo do notebook
DIAG = []

print(f"Consolidado: {len(df)} linhas — OK")
print(f"Matrizes SHAP carregadas: {len(shap_matrices)} (shapes únicos: {set(shapes.values())})")

## Seção 2 — Panorama 1: heatmap de ranks  *(§2.2, decisão D-E3-05)*

**O que checar.** Existe alguma célula em que uma feature de `S_6` receba rank ≥ 7 (foi colocada fora do top-6 por algum modelo)? Existe alguma em que `P1` ou `T2` receba rank ≤ 3 (foi promovida indevidamente)?

**Por que importa.** O heatmap é o teste visual mais barato de estabilidade de ranking. Linhas horizontais quase homogêneas indicam que os 5 modelos concordam. PA_3.1 §1.2 já reportou Spearman médio ≥ 0.96 entre modelos — anomalia aqui seria surpresa.

**O que invalidaria a leitura otimista.** Qualquer anomalia listada acima → registrar D-E3-05 como observada e considerar manter a feature em S.

**Como interpretar o resultado.** Cada célula mostra o rank da feature (coluna) atribuído por aquele modelo (linha) — `1` = feature mais importante para aquele modelo, `8` = menos. A escala `viridis_r` torna ranks baixos (top) mais escuros e ranks altos (cauda) mais claros; visualmente, cada subplot deve apresentar uma região escura concentrada nas colunas de `S_6` e uma região clara nas colunas de `P1`/`T2`.

- **Coluna verticalmente homogênea** (mesmo rank em todos os modelos) = concordância entre os 5 modelos; é o sinal qualitativo da estabilidade que PA_3.1 §1.2 quantifica em ρ̄ ≥ 0.96.
- **Coluna heterogênea dentro de `S_6`** (ex.: 4 modelos atribuem rank baixo, 1 atribui rank ≥ 7) → célula destacada com **borda vermelha**; abre risco de D-E3-05.
- **Coluna heterogênea em `P1` / `T2`** no sentido oposto (algum modelo atribui rank ≤ 3) → borda vermelha; questiona a hipótese de descarte.
- **Comparação entre os 3 subplots (outputs):** se a faixa escura de `S_6` se mantém estável de `ET` para `M_CH3OH` e para `x_CH3OH`, a granularidade global (D-E3-04) é defensável; se um output mostra padrão muito diferente, é evidência para reabrir a discussão de granularidade por output.
- **Leitura final:** zero bordas vermelhas → `status_D05 = resolvida` impresso pela célula seguinte; bordas presentes → lista de anomalias é impressa e `status_D05 = observada`, registrando D-E3-05 como pendência.

In [ ]:
def rank_table(output: str) -> pd.DataFrame:
    return (
        df[df["output"] == output]
          .pivot(index="modelo", columns="feature", values="shap_rank")
          .reindex(index=MODELS, columns=FEATURE_NAMES)
    )

anomalias = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, out in zip(axes, OUTPUTS):
    rk = rank_table(out)
    im = ax.imshow(rk.values, cmap="viridis_r", aspect="auto", vmin=1, vmax=8)
    ax.set_xticks(range(len(FEATURE_NAMES)), FEATURE_NAMES, rotation=45)
    ax.set_yticks(range(len(MODELS)), MODELS)
    ax.set_title(f"Ranks SHAP — {out}")
    for i, m in enumerate(MODELS):
        for j, f in enumerate(FEATURE_NAMES):
            r = int(rk.values[i, j])
            ax.text(j, i, r, ha="center", va="center",
                    color="white" if r <= 4 else "black", fontsize=9)
            anomalo = (f in S_6 and r >= 7) or (f in F_DESCARTADAS and r <= 3)
            if anomalo:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="red", lw=2))
                anomalias.append({"output": out, "modelo": m, "feature": f, "rank": r})
plt.tight_layout()
plt.show()

if anomalias:
    print("\n[!] Anomalias de ranking encontradas:")
    print(pd.DataFrame(anomalias).to_string(index=False))
    status_D05 = "observada"
else:
    print("\nNenhuma feature de S_6 com rank >= 7; nenhuma feature descartada (P1, T2) com rank <= 3.")
    status_D05 = "resolvida"

DIAG.append({
    "decisao":   "D-E3-05",
    "descricao": "instabilidade de ranking próximo do corte",
    "status":    status_D05,
    "evidencia": f"{len(anomalias)} célula(s) anômala(s) no heatmap (S_6 com rank>=7 ou P1/T2 com rank<=3)",
})
print(f"\n→ D-E3-05 = {status_D05}")

## Seção 3 — Panorama 2: curva de cobertura  *(§2.3, escalação de S_5?)*

**O que checar.** Onde está o cotovelo da curva? Se em k=6, S_6 é a escolha natural. Se em k=5, k=5 vira candidato sério em 3.3.

**Como interpretar.** A curva é `cobertura(k) = soma das top-k importâncias relativas normalizadas`, no **pior caso entre modelos** (para ser conservador). PA_3.1 §1.2 reporta k=6 atingindo 95% em todos os outputs.

**Como interpretar o resultado.** Cada curva mostra, para um dado output, a fração acumulada de `|SHAP|` capturada pelas top-k features — tomando o **mínimo entre os 5 modelos** em cada k (envelope conservador: se a curva passa por (k, y), então todos os modelos cobrem pelo menos `y` com k features). O eixo-x vai de 1 a 8 (todas as features); a linha pontilhada cinza em `y = 0.95` é o threshold operacional do plano e as linhas verticais marcam os k candidatos (k=6 em vermelho como hipótese principal).

- **Cotovelo nítido em k=6** (curva sobe rápido até k=6 e achata depois) → S_6 é o ponto natural de corte; é o cenário esperado por PA_3.1 §1.2.
- **Cotovelo antecipado em k=5** (curva já está quase plana entre k=5 e k=6) → S_5 ganha força como candidato em 3.3, mesmo que k=6 permaneça como hipótese principal aqui.
- **Cotovelo atrasado (k≥7)** ou alguma curva cruzando 95% só depois de k=6 → evidência contra S_6; reabrir granularidade global (D-E3-04) e considerar S_7.
- **Curvas dos 3 outputs próximas entre si** = comportamento de cobertura homogêneo, reforça granularidade global; **uma curva destoante** (ex.: `x_CH3OH` precisando de mais features para atingir 95%) é argumento para granularidade por output.
- **Margem em k=6** (distância vertical entre a curva e a linha 95%): margem confortável (≥ 2-3 p.p.) indica robustez; margem apertada (curva tangenciando 95%) sinaliza que pequenas variações no SHAP poderiam mover o cotovelo e merece ser anotado.
- **Leitura final:** a célula seguinte imprime `k_95 (pior)` e `k_99 (pior)` por output e classifica o cenário em uma das três falas (`cotovelo_lida`): k=5 já basta → escalar S_5; k=6 é o cotovelo → S_6 confirmado; k_95 > 6 → S_6 insuficiente.

In [ ]:
def cobertura_curve(modelo: str, output: str) -> np.ndarray:
    sub = df[(df["modelo"] == modelo) & (df["output"] == output)]
    s = np.sort(sub["shap_mean"].values)[::-1]
    return np.cumsum(s) / s.sum()

ks = np.arange(1, 9)
fig, ax = plt.subplots(figsize=(8, 5))
for out, color in zip(OUTPUTS, ["tab:blue", "tab:orange", "tab:green"]):
    # pior caso entre modelos para cada k
    curvas = np.stack([cobertura_curve(m, out) for m in MODELS])
    pior = curvas.min(axis=0)
    ax.plot(ks, pior, marker="o", color=color, label=f"{out} (pior caso)")

ax.axhline(0.95, color="gray", lw=1, ls=":", label="95%")
for k_marker, ls in zip([4, 5, 6], ["--", "--", "-"]):
    ax.axvline(k_marker, color="red" if k_marker == 6 else "gray", lw=1, ls=ls, alpha=0.6)
ax.set_xlabel("k (nº de features)")
ax.set_ylabel("cobertura acumulada de |SHAP|")
ax.set_title("Curva de cobertura — pior caso entre os 5 modelos")
ax.set_xticks(ks)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Diagnóstico textual: onde está o k mínimo para 95% e 99%?
rows = []
for out in OUTPUTS:
    curvas = np.stack([cobertura_curve(m, out) for m in MODELS])
    pior = curvas.min(axis=0)
    k_95 = int(np.searchsorted(pior, 0.95) + 1)
    k_99 = int(np.searchsorted(pior, 0.99) + 1)
    rows.append({"output": out, "k_95 (pior)": k_95, "k_99 (pior)": k_99})
cov_resumo = pd.DataFrame(rows)
print(cov_resumo.to_string(index=False))

k95_max = cov_resumo["k_95 (pior)"].max()
if k95_max <= 5:
    cotovelo_lida = "k=5 já cobre 95% no pior caso — escalar S_5 como candidato prioritário em 3.3"
elif k95_max == 6:
    cotovelo_lida = "k=6 é o cotovelo natural — S_6 confirmado como hipótese principal"
else:
    cotovelo_lida = f"k_95 (pior) = {k95_max}; verificar se S_6 ainda é suficiente"

DIAG.append({
    "decisao":   "cobertura",
    "descricao": "localização do cotovelo da curva de cobertura",
    "status":    "informativa",
    "evidencia": cotovelo_lida,
})
print(f"\n→ {cotovelo_lida}")

## Seção 4 — Panorama 3: timing do KernelExplainer (ANN)  *(§2.4, decisão D-E3-02)*

**O que checar.** Algum (ANN, output) > 600 s (10 min) — limite de PA_3.1 §2.4 para acionar fallback para `DeepExplainer`.

**Fonte.** Métrica `runtime_seconds` registrada pelo notebook 3.1 em cada run `phase=shap` do experimento `reduzido`.

**Como interpretar o resultado.** A célula seguinte produz duas saídas complementares: (i) uma **tabela `timing_pivot`** com o tempo de SHAP em segundos para cada uma das 15 combinações (modelo × output), útil como sanity check geral; e (ii) um **bar plot horizontal** focado apenas nas três barras da ANN (uma por output), com uma linha vertical vermelha tracejada em `600 s` (= `LIMITE_S`) marcando o orçamento de tempo de PA_3.1 §2.4.

- **Todas as barras da ANN à esquerda da linha vermelha** → KernelExplainer é viável; `status_D02 = resolvida`, decisão fechada e Fase 1 pode prosseguir sem reexecução.
- **Alguma barra cruza a linha vermelha** → orçamento estourado; `status_D02 = fallback_pendente`, e a Seção 12 (checklist) **não passa** até que 3.1 seja reexecutado trocando KernelExplainer por DeepExplainer (válido para ANN porque é diferenciável).
- **Barras curtas e parecidas entre os 3 outputs** = comportamento esperado do KernelExplainer (custo dominado por `nsamples`, não pelo output) e indica que não há output patológico.
- **Uma barra muito maior que as outras duas** = sinaliza um output com superfície de resposta mais complexa para a ANN; mesmo se < 600 s, vale registrar como nota — pode reaparecer como gargalo em 3.3 (45 runs) ou 3.4.
- **Sanity check cruzado** na tabela `timing_pivot`: SVR/DT/RF/XGBoost devem ser ordens de magnitude mais rápidos que ANN (segundos vs. minutos); se algum modelo tree-based aparecer com tempos próximos ao da ANN, é sinal de problema na configuração do explainer (provavelmente usaram KernelExplainer em vez de TreeExplainer em 3.1) e a planilha consolidada deve ser auditada.
- **Leitura final:** a linha `→ D-E3-02 = ...` impressa pela célula traz o veredito formal — `resolvida` (KernelExplainer suficiente) ou `fallback_pendente` (trocar para DeepExplainer antes de reexecutar) — e esse status alimenta diretamente o item 3 do checklist da Seção 12.

In [ ]:
import mlflow
mlflow.set_tracking_uri(MLRUNS_URI)

runs = mlflow.search_runs(experiment_names=["reduzido"], filter_string="tags.phase = 'shap'")
assert len(runs) == 15, f"Esperado 15 runs phase=shap, encontrado {len(runs)}"

timing = runs[["tags.model", "tags.output", "metrics.runtime_seconds"]].copy()
timing.columns = ["modelo", "output", "segundos"]
timing_pivot = (timing.pivot(index="modelo", columns="output", values="segundos")
                       .reindex(index=MODELS, columns=OUTPUTS).round(1))
print("Tempo (s) por (modelo, output):")
print(timing_pivot.to_string())

ann_max = timing[timing["modelo"] == "ANN"]["segundos"].max()
LIMITE_S = 600.0
if ann_max < LIMITE_S:
    status_D02 = "resolvida"
    msg_D02 = f"max(runtime_ANN) = {ann_max:.1f}s < {LIMITE_S}s — KernelExplainer suficiente"
else:
    status_D02 = "fallback_pendente"
    msg_D02 = f"max(runtime_ANN) = {ann_max:.1f}s >= {LIMITE_S}s — trocar para DeepExplainer antes de reexecutar"

fig, ax = plt.subplots(figsize=(7, 2.5))
ann_times = timing_pivot.loc["ANN"]
ax.barh(ann_times.index, ann_times.values, color="tab:purple")
ax.axvline(LIMITE_S, color="red", ls="--", lw=1, label=f"limite {LIMITE_S:.0f}s")
ax.set_xlabel("segundos")
ax.set_title("Tempo SHAP — ANN × output (D-E3-02)")
ax.legend()
plt.tight_layout()
plt.show()

DIAG.append({
    "decisao":   "D-E3-02",
    "descricao": "ANN com KernelExplainer dentro do orçamento de tempo",
    "status":    status_D02,
    "evidencia": msg_D02,
})
print(f"\n→ D-E3-02 = {status_D02} ({msg_D02})")

## Seção 5 — Detalhe: bar + beeswarm de ANN × XGBoost  *(§2.1, qualitativo)*

**O que checar.** Para os dois melhores baselines (R² ≥ 0.92): o ranking do top-3 no bar plot coincide com o eixo-y do beeswarm? A direção do efeito (cor azul/vermelho) é fisicamente plausível?

**Sinal de alerta.** Beeswarm com nuvem cinza compacta para a top-1 → feature dominante com efeito não-monotônico (não invalida o ranking, mas vira nota de discussão).

**Como interpretar o resultado.** As duas células seguintes mostram dois mosaicos 2×3 (linhas = ANN/XGBoost, colunas = ET/M_CH3OH/x_CH3OH): primeiro os **bar plots** (`|SHAP| médio` por feature, ordenado decrescente — quantifica importância) e depois os **beeswarm plots** (uma faixa horizontal por feature, cada ponto é uma observação; posição no eixo-x = valor SHAP daquela observação, cor azul→vermelho = valor da feature de baixo→alto). Os 6 pares são imagens PNG geradas em 3.1, exibidas aqui sem recomputação. Logo depois, a célula de texto imprime o top-3 numérico por par para conferência cruzada com os eixos-y dos PNGs.

- **Top-3 do bar coincide com eixo-y do beeswarm** = consistência garantida por construção (mesma fonte de dados); se divergir, há bug em 3.1 e a planilha consolidada precisa ser regerada.
- **Beeswarm com gradiente azul→vermelho da esquerda para direita** numa feature top → efeito **monotônico positivo** (valores altos da feature empurram a predição para cima); espera-se isso para RRC1/BRC1/RRC2/BRC2 sobre `ET` (mais refluxo/boil-up = mais energia).
- **Gradiente vermelho→azul** = efeito **monotônico negativo** (valores altos puxam predição para baixo); plausível por exemplo para razões de refluxo sobre `M_CH3OH` (mais refluxo = menos vazão líquida de produto).
- **Cores embaralhadas ao longo da faixa** (sem gradiente claro) = efeito **não-monotônico** ou dominado por interações; é o "sinal de alerta" do bloco acima — vira nota de discussão se aparecer na top-1, mas não invalida o ranking.
- **Discrepância ANN vs XGBoost no top-3** (ex.: ANN coloca BRC2 no top-3 e XGBoost coloca RRC2) = informação útil — feature está perto do corte e a importância depende da família do modelo; reforça por que `S_6` foi definido por união/agregação, não por um modelo só.
- **Coerência entre os 3 outputs do mesmo modelo:** se ANN mantém RRC1/BRC1 no top-3 dos três outputs, é evidência adicional de que essas features dominam o comportamento de toda a coluna 1; padrão diferente por output sugere que os outputs respondem a sub-conjuntos distintos.
- **Plausibilidade física por output:** `ET` (energia) deve responder fortemente a refluxo/boil-up das duas colunas; `M_CH3OH` (vazão) a RFF (purga) e refluxos; `x_CH3OH` (pureza) a refluxos e T1. Direções que contrariem esse senso comum vão para a discussão da Seção 6.
- **Leitura final:** essa seção é **qualitativa** (`status = qualitativa` no `DIAG`); não bloqueia o checklist da Seção 12 nem produz decisão formal. A saída esperada é uma anotação curta confirmando que as direções fazem sentido — ou listando exceções para o capítulo de discussão do TCC.

In [ ]:
TOP_MODELS = ["ANN", "XGBoost"]

# Bar plots (3.1 já gerou as PNGs — apenas exibimos)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, m in enumerate(TOP_MODELS):
    for j, o in enumerate(OUTPUTS):
        ax = axes[i, j]
        img = mpimg.imread(SHAP_BAR_DIR / f"3.1_bar_{m}_{o}.png")
        ax.imshow(img)
        ax.set_title(f"bar — {m} × {o}", fontsize=10)
        ax.axis("off")
plt.suptitle("Bar plots — |SHAP| médio por feature", y=1.01)
plt.tight_layout()
plt.show()

# Beeswarm plots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, m in enumerate(TOP_MODELS):
    for j, o in enumerate(OUTPUTS):
        ax = axes[i, j]
        img = mpimg.imread(SHAP_BEE_DIR / f"3.1_beeswarm_{m}_{o}.png")
        ax.imshow(img)
        ax.set_title(f"beeswarm — {m} × {o}", fontsize=10)
        ax.axis("off")
plt.suptitle("Beeswarm plots — distribuição de SHAP por feature", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Checklist textual: coerência interna do top-3 entre bar e beeswarm para os 6 pares (2 modelos × 3 outputs).
# Como os PNGs do beeswarm já estão ordenados por |SHAP| médio (mesma fonte de dados que o bar), a coerência interna é
# garantida por construção. O que precisa ser verificado VISUALMENTE é (i) a direção do efeito e (ii) presença de
# nuvens compactas. Imprimimos o top-3 numérico para conferência cruzada com os PNGs acima.

print("Top-3 |SHAP| médio por (modelo, output) — confira com o eixo-y dos beeswarms acima:\n")
for m in TOP_MODELS:
    for o in OUTPUTS:
        sub = df[(df["modelo"] == m) & (df["output"] == o)].nsmallest(3, "shap_rank")
        top3 = list(sub.sort_values("shap_rank")["feature"])
        print(f"  {m:8s} × {o:8s} → {top3}")

DIAG.append({
    "decisao":   "§2.1",
    "descricao": "coerência bar/beeswarm ANN × XGBoost",
    "status":    "qualitativa",
    "evidencia": "top-3 numérico impresso; confirmar visualmente direção do efeito nos beeswarms",
})

## Seção 6 — Narrativa: por que P1 e T2 ficam de fora  *(§2.5)*

**Estado numérico.** PA_3.1 §1.2 reporta:
- `P1`: max 2.6% (SVR × M_CH3OH), média 1.4%
- `T2`: max 1.8% (SVR × x_CH3OH), média 0.6%

Ambas ≪ 5% em todos os pares (threshold operacional do plano).

**Hipótese física.** P1 é a pressão do reator (50–100 bar) e termodinamicamente seria a variável mais relevante para a conversão na síntese de metanol. Que SHAP a coloque consistentemente em rank 7 não é trivial — a explicação provável é que os três outputs (`Methanol Flow`, `Purity`, `Energy`) são medidos **depois da coluna de destilação**, que re-equaliza vazão e pureza. O efeito de P1 sobre a conversão no reator existe, mas é amortecido a jusante. T2 (85–95 °C) tem range estreito, o que reduz sua variância e, por consequência, sua importância marginal.

Esta justificativa precisa estar **escrita** antes da defesa da redução.

In [ ]:
# Recomputa a contribuição relativa de cada feature por (modelo, output) para confirmar os números de PA_3.1 §1.2.
df_rel = df.copy()
df_rel["shap_rel"] = (
    df_rel.groupby(["modelo", "output"])["shap_mean"]
          .transform(lambda x: x / x.sum())
)

contrib_descartadas = (
    df_rel[df_rel["feature"].isin(F_DESCARTADAS)]
          .groupby("feature")["shap_rel"]
          .agg(["max", "mean"])
          .reindex(F_DESCARTADAS)
)
print("Contribuição relativa de P1 e T2 (em todos os 15 pares modelo × output):\n")
print((contrib_descartadas * 100).round(2).astype(str) + " %")

# Onde cada feature descartada atinge sua contribuição máxima
for f in F_DESCARTADAS:
    row = df_rel[df_rel["feature"] == f].nlargest(1, "shap_rel").iloc[0]
    print(f"  {f}: max em {row['modelo']} × {row['output']} = {row['shap_rel']*100:.2f} %")

THRESHOLD = 0.05
assert contrib_descartadas["max"].max() < THRESHOLD, "Alguma feature descartada > 5% — revisar S_6"
print(f"\n→ Ambas as features descartadas têm contribuição relativa < {THRESHOLD*100:.0f}% em todos os pares.")

DIAG.append({
    "decisao":   "§2.5",
    "descricao": "contribuição relativa máxima de P1, T2",
    "status":    "resolvida",
    "evidencia": f"P1 max = {contrib_descartadas.loc['P1','max']*100:.2f}% ; T2 max = {contrib_descartadas.loc['T2','max']*100:.2f}% (< {THRESHOLD*100:.0f}%)",
})

## Seção 7 — Painel de síntese

Figura única (3 sub-plots) que resume o argumento de S_6 em um quadro: heatmap de ranks com S_6 destacado + curva de cobertura + bar plot da importância global agregada.

Um leitor que pule §1–§6 deve conseguir ler **só esta figura** e entender por que S_6.

In [ ]:
# Importância global agregada (recomputo idêntico ao previsto em 3.2 §3)
importancia_global = (
    df_rel.groupby("feature")["shap_rel"].mean()
          .sort_values(ascending=False)
)
top6_global = list(importancia_global.head(6).index)

fig = plt.figure(figsize=(16, 5))
gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 1.0])

# (a) Heatmap consolidado: rank médio entre modelos, por (output, feature), com bordas em S_6
ax_a = fig.add_subplot(gs[0, 0])
rank_avg = (
    df.groupby(["output", "feature"])["shap_rank"].mean()
      .unstack("feature").reindex(index=OUTPUTS, columns=FEATURE_NAMES)
)
im = ax_a.imshow(rank_avg.values, cmap="viridis_r", aspect="auto", vmin=1, vmax=8)
ax_a.set_xticks(range(len(FEATURE_NAMES)), FEATURE_NAMES, rotation=45)
ax_a.set_yticks(range(len(OUTPUTS)), OUTPUTS)
ax_a.set_title("(a) Rank SHAP médio (modelos)\nbordas vermelhas = S_6")
for i in range(len(OUTPUTS)):
    for j, f in enumerate(FEATURE_NAMES):
        ax_a.text(j, i, f"{rank_avg.values[i,j]:.1f}", ha="center", va="center",
                  color="white" if rank_avg.values[i,j] <= 4 else "black", fontsize=9)
        if f in S_6:
            ax_a.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="red", lw=2))
fig.colorbar(im, ax=ax_a, shrink=0.7, label="rank médio")

# (b) Cobertura pior caso (3 curvas), linhas em k=6 e 95%
ax_b = fig.add_subplot(gs[0, 1])
for out, color in zip(OUTPUTS, ["tab:blue", "tab:orange", "tab:green"]):
    curvas = np.stack([cobertura_curve(m, out) for m in MODELS])
    ax_b.plot(ks, curvas.min(axis=0), marker="o", color=color, label=out)
ax_b.axhline(0.95, color="gray", lw=1, ls=":")
ax_b.axvline(6, color="red", lw=1.2)
ax_b.set_xlabel("k")
ax_b.set_ylabel("cobertura (pior caso)")
ax_b.set_title("(b) Cobertura por output")
ax_b.set_xticks(ks)
ax_b.set_ylim(0, 1.02)
ax_b.legend(loc="lower right", fontsize=9)
ax_b.grid(alpha=0.3)

# (c) Importância global agregada
ax_c = fig.add_subplot(gs[0, 2])
colors = ["tab:red" if f in S_6 else "lightgray" for f in importancia_global.index]
ax_c.barh(importancia_global.index[::-1],
          importancia_global.values[::-1],
          color=colors[::-1], edgecolor="black", lw=0.5)
ax_c.set_xlabel("importância global (∑ shap_rel / 15)")
ax_c.set_title("(c) Importância global — vermelho = S_6")
ax_c.grid(axis="x", alpha=0.3)

plt.suptitle("Mapa SHAP — síntese do argumento para S_6 = {T1, RRC1, BRC1, RRC2, BRC2, RFF}",
             y=1.03, fontsize=12)
plt.tight_layout()

painel_path = OUT_DIR / "3.1.2_painel_S6.png"
fig.savefig(painel_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"\nSalvo: {painel_path}")

## Seção 8 — Decisão A: S_k principal  *(PA_3.1 §3.1, D-E3-04)*

Três critérios independentes convergem:
- Top-k fixo (PLANO_3) varrendo k ∈ {4, 5, 6}
- Cobertura 95% — pior caso k=6 em todos os outputs
- União estável de Top-k entre modelos — estabiliza em k=4 com |U|=6

**Conclusão.** `S_6 = {T1, RRC1, BRC1, RRC2, BRC2, RFF}` é a hipótese principal a entrar em 3.3.

In [ ]:
print("D-E3-04 (atualizada): granularidade global, S_6 como hipótese central.")
print(f"  S_6 = {S_6}")
print(f"  Descartadas: {F_DESCARTADAS}")

DIAG.append({
    "decisao":   "D-E3-04",
    "descricao": "granularidade global; S_6 como hipótese principal",
    "status":    "definida",
    "evidencia": f"S_6 = {S_6}; convergência de 3 critérios (top-k fixo, cobertura 95%, união estável)",
})

## Seção 9 — Decisão B: manter k=4 e k=5 na varredura  *(PA_3.1 §3.2, D-E3-04b)*

**Pergunta.** Manter k ∈ {4, 5, 6} na varredura de 3.3 (45 runs) ou reduzir a k=6 (15 runs)?

**Resposta.** Manter. O lugar canônico para decidir até onde a redução aguenta é o teste de equivalência (3.4). Cortar agora seria contaminar a decisão com uma intuição baseada em SHAP médio (que já vimos ser uma agregação grosseira).

**Atenção.** Para k=4 e k=5, S_k deixa de ser único entre os critérios — manter a regra `top-k por importância global` de PLANO_3 §3.2.

In [ ]:
# Preview dos S_k que 3.2 vai materializar (top-k da importância global)
S_preview = {k: list(importancia_global.head(k).index) for k in [4, 5, 6]}
for k, sk in S_preview.items():
    print(f"  S_{k} = {sk}")

DIAG.append({
    "decisao":   "D-E3-04b",
    "descricao": "varredura k ∈ {4, 5, 6} mantida",
    "status":    "definida",
    "evidencia": f"S_4 = {S_preview[4]}; S_5 = {S_preview[5]}; S_6 = {S_preview[6]}",
})

## Seção 10 — Decisão C: top-k global × união estável  *(PA_3.1 §3.3, D-E3-11)*

**Pergunta.** S_6 (união estável de 3.1.1) e S_6 (top-6 da importância global, regra do PLANO_3) coincidem?

**Importância.** Esta é a única decisão **falsificável** aqui. Se coincidem, é dupla validação metodológica para o TCC. Se divergem, prevalece o `top-k global` (regra original) e a divergência vira material de discussão.

In [ ]:
set_global  = set(top6_global)
set_uniao   = set(S_6)
convergente = set_global == set_uniao

print(f"Top-6 (importância global): {sorted(set_global, key=FEATURE_NAMES.index)}")
print(f"União estável (3.1.1):      {sorted(set_uniao,  key=FEATURE_NAMES.index)}")
print(f"Divergência: {set_global.symmetric_difference(set_uniao) or '(vazia)'}")

if convergente:
    status_D11 = "convergente"
    msg_D11 = "Dupla validação: top-k global e união estável coincidem em S_6."
else:
    status_D11 = "divergente"
    msg_D11 = (
        f"Divergência entre métodos: top-k global = {sorted(set_global, key=FEATURE_NAMES.index)}; "
        f"união estável = {sorted(set_uniao, key=FEATURE_NAMES.index)}. "
        "Prevalece top-k global (PLANO_3 §3.2)."
    )

print(f"\n→ D-E3-11 = {status_D11}\n  {msg_D11}")

DIAG.append({
    "decisao":   "D-E3-11",
    "descricao": "comparação top-k global × união estável",
    "status":    status_D11,
    "evidencia": msg_D11,
})

## Seção 11 — Decisão D: consolidar D-E3-02 e D-E3-05  *(PA_3.1 §3.4)*

Já populadas nas Seções 2 e 4. Aqui apenas releitura.

In [ ]:
diag_df = pd.DataFrame(DIAG)
print("Estado consolidado das decisões da Fase 1:\n")
print(diag_df.to_string(index=False))

diag_path = OUT_DIR / "3.1.2_diagnostico_decisoes.csv"
diag_df.to_csv(diag_path, index=False)
print(f"\nSalvo: {diag_path}")

## Seção 12 — Checklist de fechamento da Fase 1  *(PA_3.1 §4.2)*

Verificação programática dos 6 itens. Se todos passarem, a Fase 1 está concluída e o próximo passo é disparar `3.2_feature_selection.ipynb`.

In [ ]:
checks = []

# 1) consolidado com 120 linhas, sem NaN
checks.append((
    "3.1_shap_consolidado.csv com 120 linhas sem NaN",
    len(df) == 120 and not df.isna().any().any(),
))

# 2) 15 runs MLflow phase=shap
checks.append((
    "15 runs MLflow phase=shap registradas",
    len(runs) == 15,
))

# 3) D-E3-02 resolvida ou com fallback executado
checks.append((
    "D-E3-02 (ANN com KernelExplainer)",
    status_D02 == "resolvida",
))

# 4) D-E3-04 atualizada com S_6 explícito
checks.append((
    "D-E3-04 atualizada com S_6 explícito",
    "D-E3-04" in diag_df["decisao"].values and len(S_6) == 6,
))

# 5) D-E3-05 resolvida ou anotada com mitigação
checks.append((
    "D-E3-05 (instabilidade de ranking)",
    status_D05 == "resolvida",
))

# 6) Parágrafo de plausibilidade de descarte de P1 escrito
checks.append((
    "Parágrafo §2.5 (plausibilidade física do descarte de P1, T2) escrito no notebook",
    True,  # presente em Seção 6 (markdown)
))

for label, passed in checks:
    print(f"  [{'x' if passed else ' '}] {label}")

if all(p for _, p in checks):
    print("\n=" * 1 + "=" * 71)
    print(" FASE 1 ENCERRADA — disparar 3.2_feature_selection.ipynb")
    print("=" * 72)
else:
    pendentes = [label for label, p in checks if not p]
    print(f"\n[!] Itens pendentes ({len(pendentes)}):")
    for label in pendentes:
        print(f"    - {label}")

## Seção 13 — Exportação dos outputs

Sem logging em MLflow. Esta seção apenas consolida o que foi impresso nas células anteriores em um único arquivo `3.1.2_CELL_OUTPUTS.md` na pasta da etapa. As figuras já foram salvas onde foram geradas (`3.1.2_painel_S6.png` em Seção 7).

In [ ]:
from datetime import datetime
import io

md = io.StringIO()
def w(s: str = "") -> None:
    md.write(s + "\n")

w("# 3.1.2 — Outputs das células")
w(f"\n_Gerado automaticamente em {datetime.now().strftime('%Y-%m-%d %H:%M')}_")
w("_Fonte: `ARTEFATOS/ETAPA_3/3.1.2_mapa_shap.ipynb`_\n")

w("## Seção 1 — Setup e carga\n")
w("```")
w(f"Consolidado: {len(df)} linhas — OK")
w(f"Matrizes SHAP carregadas: {len(shap_matrices)} (shapes únicos: {set(shapes.values())})")
w("```\n")

w("## Seção 2 — Heatmap de ranks (D-E3-05)\n")
w("```")
if anomalias:
    w("[!] Anomalias de ranking encontradas:")
    w(pd.DataFrame(anomalias).to_string(index=False))
else:
    w("Nenhuma feature de S_6 com rank >= 7; nenhuma feature descartada (P1, T2) com rank <= 3.")
w("")
w(f"→ D-E3-05 = {status_D05}")
w("```\n")

w("## Seção 3 — Curva de cobertura\n")
w("```")
w(cov_resumo.to_string(index=False))
w("")
w(f"→ {cotovelo_lida}")
w("```\n")

w("## Seção 4 — Timing ANN (D-E3-02)\n")
w("```")
w("Tempo (s) por (modelo, output):")
w(timing_pivot.to_string())
w("")
w(f"→ D-E3-02 = {status_D02} ({msg_D02})")
w("```\n")

w("## Seção 5 — Top-3 SHAP ANN × XGBoost\n")
w("```")
w("Top-3 |SHAP| médio por (modelo, output) — confira com o eixo-y dos beeswarms:")
w("")
for m in TOP_MODELS:
    for o in OUTPUTS:
        sub = df[(df["modelo"] == m) & (df["output"] == o)].nsmallest(3, "shap_rank")
        top3 = list(sub.sort_values("shap_rank")["feature"])
        w(f"  {m:8s} × {o:8s} → {top3}")
w("```\n")

w("## Seção 6 — Descarte de P1 e T2\n")
w("```")
w("Contribuição relativa de P1 e T2 (em todos os 15 pares modelo × output):")
w("")
w(((contrib_descartadas * 100).round(2).astype(str) + " %").to_string())
for f in F_DESCARTADAS:
    row = df_rel[df_rel["feature"] == f].nlargest(1, "shap_rel").iloc[0]
    w(f"  {f}: max em {row['modelo']} × {row['output']} = {row['shap_rel']*100:.2f} %")
w("")
w(f"→ Ambas as features descartadas têm contribuição relativa < {THRESHOLD*100:.0f}% em todos os pares.")
w("```\n")

w("## Seção 7 — Painel de síntese\n")
w(f"Figura salva em: `{painel_path.relative_to(PROJECT_ROOT)}`\n")

w("## Seção 8 — Decisão A (D-E3-04)\n")
w("```")
w("D-E3-04 (atualizada): granularidade global, S_6 como hipótese central.")
w(f"  S_6 = {S_6}")
w(f"  Descartadas: {F_DESCARTADAS}")
w("```\n")

w("## Seção 9 — Decisão B (D-E3-04b): varredura k ∈ {4, 5, 6}\n")
w("```")
for k, sk in S_preview.items():
    w(f"  S_{k} = {sk}")
w("```\n")

w("## Seção 10 — Decisão C (D-E3-11)\n")
w("```")
w(f"Top-6 (importância global): {sorted(set_global, key=FEATURE_NAMES.index)}")
w(f"União estável (3.1.1):      {sorted(set_uniao,  key=FEATURE_NAMES.index)}")
w(f"Divergência: {set_global.symmetric_difference(set_uniao) or '(vazia)'}")
w("")
w(f"→ D-E3-11 = {status_D11}")
w(f"  {msg_D11}")
w("```\n")

w("## Seção 11 — Estado consolidado das decisões\n")
w("```")
w("Estado consolidado das decisões da Fase 1:")
w("")
w(diag_df.to_string(index=False))
w("```\n")
w(f"CSV salvo em: `{diag_path.relative_to(PROJECT_ROOT)}`\n")

w("## Seção 12 — Checklist de fechamento da Fase 1\n")
w("```")
for label, passed in checks:
    w(f"  [{'x' if passed else ' '}] {label}")
if all(p for _, p in checks):
    w("")
    w("=" * 72)
    w(" FASE 1 ENCERRADA — disparar 3.2_feature_selection.ipynb")
    w("=" * 72)
else:
    pendentes = [label for label, p in checks if not p]
    w("")
    w(f"[!] Itens pendentes ({len(pendentes)}):")
    for label in pendentes:
        w(f"    - {label}")
w("```\n")

outputs_path = OUT_DIR / "3.1.2_CELL_OUTPUTS.md"
outputs_path.write_text(md.getvalue(), encoding="utf-8")
print(f"Salvo: {outputs_path}")